# 🧠 Keras — Complete Revision Guide

> **Keras** is a high-level deep learning API built on top of TensorFlow 2.x.  
> It simplifies building, training, and deploying neural networks.

---

## 📚 What You Will Learn
1. Setup & Core Concepts  
2. Building Models (Sequential / Functional / Subclass)  
3. Layers, Activations, Loss, Optimizers  
4. Training, Evaluation, Callbacks  
5. CNN — Image Classification on MNIST  
6. LSTM — Sentiment-like Sequence Model  
7. Transfer Learning  
8. Custom Layers & Training Loops  
9. Saving & Loading Models  
10. Interview Q&A  

---
## 1️⃣ Installation & Verification

In [ ]:
# Run this cell first to verify TensorFlow / Keras is installed
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 2️⃣ Core Concepts — Explained Simply

| Term | What it means |
|---|---|
| **Tensor** | Multi-dimensional array (like NumPy ndarray but on GPU) |
| **Layer** | Building block — applies a math transformation to data |
| **Model** | A group of layers chained together |
| **Weights** | Numbers inside layers that the model *learns* |
| **Epoch** | One full pass of ALL training data through the model |
| **Batch** | Small chunk of data processed at once (e.g., 32 samples) |
| **Gradient** | Direction to nudge weights to reduce error |
| **Loss** | How wrong the model's prediction is (we minimize this) |
| **Optimizer** | Algorithm that updates weights using gradients |

In [ ]:
# --- Tensors are just arrays ---
import tensorflow as tf

scalar  = tf.constant(42)                          # 0-D tensor
vector  = tf.constant([1.0, 2.0, 3.0])            # 1-D tensor
matrix  = tf.constant([[1, 2], [3, 4]])            # 2-D tensor
tensor3 = tf.zeros((2, 3, 4))                      # 3-D tensor

print('Scalar  :', scalar.numpy())
print('Vector  :', vector.numpy())
print('Matrix  :\n', matrix.numpy())
print('3D shape:', tensor3.shape)

---
## 3️⃣ Building Models — 3 Ways

### 📌 Way 1: Sequential API
Use when layers are stacked one after another (most common for beginners).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Sequential: input → Dense(64) → Dense(64) → output(10)
model_seq = keras.Sequential([
    layers.Input(shape=(784,)),          # Input shape = flattened 28x28 image
    layers.Dense(128, activation='relu'),  # Hidden layer 1 — 128 neurons
    layers.Dense(64,  activation='relu'),  # Hidden layer 2 — 64 neurons
    layers.Dense(10,  activation='softmax')# Output — 10 classes (digits 0-9)
], name='sequential_mlp')

model_seq.summary()
# Total params tells you how many weights this model has

### 📌 Way 2: Functional API
Use when you need **multiple inputs/outputs**, **skip connections**, or **shared layers**.

In [ ]:
# Functional API — more explicit about data flow
inputs  = keras.Input(shape=(784,), name='input_layer')
x       = layers.Dense(128, activation='relu', name='hidden1')(inputs)
x       = layers.Dropout(0.3, name='dropout1')(x)   # Drop 30% of neurons randomly
x       = layers.Dense(64,  activation='relu', name='hidden2')(x)
outputs = layers.Dense(10,  activation='softmax', name='output')(x)

model_func = keras.Model(inputs=inputs, outputs=outputs, name='functional_mlp')
model_func.summary()

# You can visualize the graph
# keras.utils.plot_model(model_func, show_shapes=True)

### 📌 Way 3: Model Subclassing
Use when you need full Python control (custom `forward` pass logic).

In [ ]:
class MyMLP(keras.Model):
    def __init__(self, hidden_units, num_classes):
        super().__init__()
        self.hidden1 = layers.Dense(hidden_units, activation='relu')
        self.hidden2 = layers.Dense(hidden_units // 2, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.hidden1(inputs)
        x = self.dropout(x, training=training)  # Only drops during training
        x = self.hidden2(x)
        return self.output_layer(x)

model_sub = MyMLP(hidden_units=128, num_classes=10)

# Build by passing a dummy input
dummy = tf.zeros((1, 784))
_ = model_sub(dummy)
model_sub.summary()

---
## 4️⃣ Key Layers Explained with Output Shapes

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# --- Dense (Fully Connected) ---
x = tf.random.normal((4, 10))         # Batch of 4, each 10 features
dense = layers.Dense(5, activation='relu')
out = dense(x)
print(f'Dense      input {x.shape} → output {out.shape}')   # (4, 5)

# --- Flatten ---
img = tf.random.normal((4, 28, 28))   # 4 grayscale images
flat = layers.Flatten()(img)
print(f'Flatten    input {img.shape} → output {flat.shape}') # (4, 784)

# --- Conv2D ---
img4d = tf.random.normal((4, 28, 28, 1))  # 4 images, 28x28, 1 channel
conv_out = layers.Conv2D(32, (3,3), activation='relu', padding='same')(img4d)
print(f'Conv2D     input {img4d.shape} → output {conv_out.shape}')  # (4,28,28,32)

# --- MaxPooling2D ---
pool_out = layers.MaxPooling2D((2,2))(conv_out)
print(f'MaxPool2D  input {conv_out.shape} → output {pool_out.shape}')# (4,14,14,32)

# --- BatchNormalization ---
bn_out = layers.BatchNormalization()(conv_out, training=True)
print(f'BatchNorm  same shape: {bn_out.shape}')

# --- Dropout ---
x2 = tf.ones((2, 8))
drop_out = layers.Dropout(0.5)(x2, training=True)
print(f'\nDropout input:\n  {x2.numpy()}')
print(f'Dropout output (50% zeroed):\n  {drop_out.numpy()}')

---
## 5️⃣ Activation Functions — Visual Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

x = np.linspace(-5, 5, 200).astype(np.float32)

activations = {
    'ReLU':    tf.nn.relu(x),
    'Sigmoid': tf.nn.sigmoid(x),
    'Tanh':    tf.nn.tanh(x),
    'ELU':     tf.nn.elu(x),
    'GELU':    tf.nn.gelu(x),
    'Softplus':tf.nn.softplus(x)
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, (name, y) in enumerate(activations.items()):
    axes[i].plot(x, y.numpy(), color='steelblue', linewidth=2)
    axes[i].axhline(0, color='gray', linewidth=0.5)
    axes[i].axvline(0, color='gray', linewidth=0.5)
    axes[i].set_title(name, fontsize=13, fontweight='bold')
    axes[i].set_ylim(-2, 5)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Activation Functions Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
When to use:
  ReLU    → Default for hidden layers (fast, simple)
  Sigmoid → Binary classification OUTPUT layer
  Softmax → Multi-class classification OUTPUT layer (not shown — outputs sum to 1)
  Tanh    → RNNs/LSTMs hidden layers
  ELU     → When you worry about dying ReLU (negative region handled)
  GELU    → Transformers (BERT, GPT)
""")

---
## 6️⃣ Loss Functions, Optimizers & Compiling

```
TASK TYPE           LOSS                             OUTPUT ACTIVATION
─────────────────────────────────────────────────────────────────────
Binary Classif.     binary_crossentropy              sigmoid
Multi-class         sparse_categorical_crossentropy  softmax (integer labels)
Multi-class         categorical_crossentropy         softmax (one-hot labels)
Regression          mean_squared_error               linear (none)
Regression          mean_absolute_error              linear (none)
```

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam, SGD

# Build a simple model
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax')
])

# Compile — links loss + optimizer + metrics together
model.compile(
    optimizer=Adam(learning_rate=0.001),     # Adam is the go-to optimizer
    loss='sparse_categorical_crossentropy',  # Integer labels → sparse version
    metrics=['accuracy']
)

print('Model compiled successfully!')
print(f'Optimizer : {model.optimizer.get_config()["name"]}')
print(f'Loss      : {model.loss}')

---
## 7️⃣ Full Training Example — MNIST Digit Classification

MNIST = 70,000 handwritten digit images (28×28 pixels, 10 classes: 0–9).  
This is the "Hello World" of deep learning.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ── Load dataset ──────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f'Train images : {X_train.shape}  Labels: {y_train.shape}')
print(f'Test  images : {X_test.shape}   Labels: {y_test.shape}')
print(f'Pixel range  : [{X_train.min()}, {X_train.max()}]')

# ── Normalize pixel values to [0, 1] ─────────────────────
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# ── Flatten 28x28 → 784 for Dense layers ─────────────────
X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

# ── Visualize some samples ────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 3))
for i in range(10):
    axes[0, i].imshow(X_train[i], cmap='gray')
    axes[0, i].set_title(f'Label: {y_train[i]}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(X_train[i+10], cmap='gray')
    axes[1, i].set_title(f'Label: {y_train[i+10]}', fontsize=9)
    axes[1, i].axis('off')
plt.suptitle('Sample MNIST Images', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Build MLP model ──────────────────────────────────────
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
], name='mnist_mlp')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Callbacks ────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# ── Train ────────────────────────────────────────────────
history = model.fit(
    X_train_flat, y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# ── Evaluate ─────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f'\nTest Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')

# ── Plot training curves ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['accuracy'],     label='Train Acc',  color='steelblue')
ax1.plot(history.history['val_accuracy'], label='Val Acc',    color='orange')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train Loss', color='steelblue')
ax2.plot(history.history['val_loss'], label='Val Loss',   color='orange')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Predictions ──────────────────────────────────────────
probs   = model.predict(X_test_flat[:20], verbose=0)  # probabilities (softmax)
preds   = probs.argmax(axis=1)                         # predicted class
actuals = y_test[:20]

print('Predicted:', preds)
print('Actual   :', actuals)
print('Correct  :', (preds == actuals).sum(), '/ 20')

# ── Visualize predictions ────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_test[i], cmap='gray')
    color = 'green' if preds[i] == actuals[i] else 'red'
    ax.set_title(f'P:{preds[i]} A:{actuals[i]}', fontsize=8, color=color)
    ax.axis('off')
plt.suptitle('Green=Correct  Red=Wrong', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8️⃣ CNN — Convolutional Neural Network on MNIST

CNNs are designed for images. Instead of Dense layers, they use **filters** (small matrices) that slide across the image to detect edges, shapes, patterns.

```
Input Image → [Conv2D → ReLU → Pool] × N → Flatten → Dense → Output
```

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# ── Reshape data for CNN: (samples, height, width, channels) ──
X_train_cnn = X_train.reshape(-1, 28, 28, 1)   # 1 = grayscale
X_test_cnn  = X_test.reshape(-1, 28, 28, 1)

print(f'CNN input shape: {X_train_cnn.shape}')  # (60000, 28, 28, 1)

# ── Build CNN ────────────────────────────────────────────
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # Block 1: detect basic edges and lines
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),          # 28x28 → 14x14

    # Block 2: detect combinations of edges (shapes)
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),          # 14x14 → 7x7

    # Block 3: detect higher-level features (digit parts)
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    # Classification head
    layers.GlobalAveragePooling2D(),    # Better than Flatten — fewer params
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='mnist_cnn')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=4, restore_best_weights=True)],
    verbose=1
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f'\nCNN Test Accuracy: {cnn_acc*100:.2f}%')
print(f'MLP Test Accuracy: {test_acc*100:.2f}%')
print(f'CNN improvement  : +{(cnn_acc - test_acc)*100:.2f}%')

---
## 9️⃣ LSTM — Sequential Data Classification

LSTMs process **sequences** (time series, text). They remember important things and forget irrelevant things via **gates**.

**Trick:** MNIST digits can also be treated as sequences of 28 rows (each row = 28 pixels).  
This demos LSTM without needing external data.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Treat each MNIST image as a sequence of 28 rows, each row = 28 pixels
X_train_seq = X_train          # (60000, 28, 28) — 28 timesteps, 28 features
X_test_seq  = X_test

lstm_model = keras.Sequential([
    layers.Input(shape=(28, 28)),

    layers.LSTM(128, return_sequences=True),  # Returns output at every timestep
    layers.Dropout(0.3),
    layers.LSTM(64, return_sequences=False),  # Returns only last timestep
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='mnist_lstm')

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test_seq, y_test, verbose=0)
print(f'\nLSTM Test Accuracy: {lstm_acc*100:.2f}%')

---
## 🔟 Custom Layers & Custom Training Loop

When you need something `keras.layers` doesn't have — build your own.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# ── Custom Layer: Linear (Dense without bias) ─────────────
class LinearLayer(keras.layers.Layer):
    """y = xW  (no bias)"""
    def __init__(self, units):
        super().__init__()
        self.units = units

    def build(self, input_shape):
        # This is called on first forward pass — creates weights lazily
        self.W = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='W'
        )

    def call(self, inputs):
        return tf.matmul(inputs, self.W)   # No bias term

# Test custom layer
layer = LinearLayer(units=4)
x = tf.random.normal((3, 8))
out = layer(x)
print(f'Custom LinearLayer: input {x.shape} → output {out.shape}')
print(f'Trainable weights : {[w.name for w in layer.trainable_weights]}')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Custom Training Loop (replaces model.fit) ─────────────
# Useful when you need full control (GANs, meta-learning, etc.)

# Build model
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10)
])

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn   = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
acc_metric = keras.metrics.SparseCategoricalAccuracy()

# Create dataset
dataset = tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
dataset = dataset.shuffle(10000).batch(256)

@tf.function   # Compile to graph for speed
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss   = loss_fn(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    acc_metric.update_state(y_batch, logits)
    return loss

# Train for 3 epochs
for epoch in range(3):
    total_loss = 0.0
    n_batches  = 0
    acc_metric.reset_state()

    for x_batch, y_batch in dataset:
        loss = train_step(x_batch, y_batch)
        total_loss += loss.numpy()
        n_batches  += 1

    avg_loss = total_loss / n_batches
    avg_acc  = acc_metric.result().numpy()
    print(f'Epoch {epoch+1}/3 — Loss: {avg_loss:.4f}  Acc: {avg_acc:.4f}')

---
## 1️⃣1️⃣ Saving & Loading Models

In [ ]:
import os

# ── Save full model ─────────────────────────────────────
save_path = '/tmp/mnist_cnn.keras'
cnn_model.save(save_path)
print(f'Model saved to: {save_path}')
print(f'File size: {os.path.getsize(save_path) / 1024:.1f} KB')

# ── Load & verify ────────────────────────────────────────
loaded_model = keras.models.load_model(save_path)
_, loaded_acc = loaded_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f'\nLoaded model accuracy: {loaded_acc*100:.2f}%')
print('Original and loaded model give same results:', 
      np.allclose(
          cnn_model.predict(X_test_cnn[:5], verbose=0),
          loaded_model.predict(X_test_cnn[:5], verbose=0)
      ))

# ── Save only weights ────────────────────────────────────
cnn_model.save_weights('/tmp/cnn_weights.weights.h5')
print('\nWeights-only saved.')

---
## 1️⃣2️⃣ Regularization — Fighting Overfitting

In [ ]:
from tensorflow.keras import regularizers

# ── Comparing Overfit vs Regularized model (on a tiny dataset) ──
# Use only 1000 samples to force overfitting
X_small = X_train_flat[:1000]
y_small = y_train[:1000]

def make_overfit_model():
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

def make_regularized_model():
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),  # L2
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])

for name, model_fn in [('Overfit', make_overfit_model), ('Regularized', make_regularized_model)]:
    m = model_fn()
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_small, y_small, epochs=30, validation_data=(X_test_flat, y_test),
              verbose=0, batch_size=64)
    tr = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    print(f'{name:15s}: Train={tr:.3f}  Val={va:.3f}  Gap={tr-va:.3f}')

print('\nSmaller gap = better generalization (less overfit)')

---
## 1️⃣3️⃣ Interview Questions & Answers

Run each cell to see the answer demonstrated in code.

In [ ]:
# Q1: What is the difference between model.predict() and model(x)?
import tensorflow as tf
import time
import numpy as np

x = X_test_flat[:1000]

# model(x) — direct __call__, eager, faster for small inputs
t0 = time.time()
out1 = cnn_model(X_test_cnn[:1000], training=False)
t1 = time.time()

# model.predict() — batches internally, best for large datasets
out2 = cnn_model.predict(X_test_cnn[:1000], verbose=0)
t2 = time.time()

print('Same results:', np.allclose(out1.numpy(), out2, atol=1e-5))
print(f'model(x) time    : {(t1-t0)*1000:.1f}ms')
print(f'model.predict()  : {(t2-t1)*1000:.1f}ms')
print('\nUse model.predict() for large batches, model(x) inside tf.function')

In [ ]:
# Q2: sparse_categorical_crossentropy vs categorical_crossentropy
import tensorflow as tf
import numpy as np

# True labels as integers
y_int  = np.array([0, 1, 2])                         # integer labels
y_onehot = tf.one_hot(y_int, depth=3).numpy()        # one-hot labels

# Predicted probabilities
y_pred = np.array([[0.9, 0.05, 0.05],
                   [0.1, 0.8,  0.1 ],
                   [0.1, 0.1,  0.8 ]], dtype=np.float32)

# Both should give same loss
loss_sparse = tf.keras.losses.SparseCategoricalCrossentropy()(y_int,    y_pred)
loss_categ  = tf.keras.losses.CategoricalCrossentropy()(y_onehot, y_pred)

print(f'Sparse (integer labels): {loss_sparse.numpy():.4f}')
print(f'Categ  (one-hot labels): {loss_categ.numpy():.4f}')
print(f'Are equal: {abs(loss_sparse.numpy() - loss_categ.numpy()) < 1e-5}')
print('\nUse sparse when labels are integers, categorical when one-hot encoded')

In [ ]:
# Q3: What does return_sequences=True do in LSTM?
import tensorflow as tf
from tensorflow.keras import layers

x = tf.random.normal((2, 5, 8))   # batch=2, seq_len=5, features=8

lstm_no_seq  = layers.LSTM(16, return_sequences=False)(x)  # Default
lstm_with_seq = layers.LSTM(16, return_sequences=True)(x)  # All timesteps

print(f'Input shape                    : {x.shape}')          # (2, 5, 8)
print(f'LSTM return_sequences=False    : {lstm_no_seq.shape}') # (2, 16) — last only
print(f'LSTM return_sequences=True     : {lstm_with_seq.shape}')# (2, 5, 16) — all
print('\nreturn_sequences=True needed when STACKING LSTM layers')

---
## 📊 Model Comparison Summary

In [ ]:
print('=' * 55)
print(f'{"Model":<20} {"Test Accuracy":>15} {"Params":>15}')
print('-' * 55)
print(f'{"MLP (Dense)":<20} {test_acc*100:>14.2f}%  {model_seq.count_params():>14,}')
print(f'{"CNN":<20} {cnn_acc*100:>14.2f}%  {cnn_model.count_params():>14,}')
print(f'{"LSTM":<20} {lstm_acc*100:>14.2f}%  {lstm_model.count_params():>14,}')
print('=' * 55)
print('\nKey Takeaways:')
print('  • CNN is best for images (spatial patterns)')
print('  • LSTM is best for sequences (temporal patterns)')
print('  • MLP (Dense) is good baseline for tabular data')

---
## ✅ Quick Reference Cheatsheet

```python
# BUILD
model = keras.Sequential([layers.Dense(64, activation='relu'), layers.Dense(10, activation='softmax')])

# COMPILE
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# TRAIN
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# EVALUATE
loss, acc = model.evaluate(X_test, y_test)

# PREDICT
probs = model.predict(X_new)
classes = probs.argmax(axis=1)

# SAVE / LOAD
model.save('model.keras')
model = keras.models.load_model('model.keras')
```

---
*Last updated: May 2026*